In [20]:
import pandas as pd

df = pd.read_csv('customer-dataset.csv')
df

,Unnamed: 0,flags,instruction,category,intent,response
0,0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...
...,...,...,...,...,...,...
26867,26867,BL,I am waiting for a rebate of {{Refund Amount}}...,REFUND,track_refund,Thank you for sharing your situation regarding...
26868,26868,BIL,how to see if there is anything wrong with my ...,REFUND,track_refund,Ensuring the accuracy of your restitution is o...
26869,26869,BLQZ,I'm waiting for a reimbjrsement of {{Currency ...,REFUND,track_refund,"Firstly, I genuinely understand the importance..."
26870,26870,BL,I don't know what to do to see my reimbursemen...,REFUND,track_refund,I've understood you're unsure about how to che...


In [21]:
def get_split_dataset(categories_selected, intents_selected, max_total_rows=60, random_state=42):
    intent_count = len(intents_selected)
    # Largest even count per intent that stays within the cap, so train and test match.
    rows_per_intent = (max_total_rows // intent_count) // 2 * 2
    rows_per_split = rows_per_intent // 2
    if rows_per_split < 1:
        raise ValueError("max_total_rows is too small for one train and one test row per intent")

    print(f"intents: {intent_count}")
    print(f"rows_per_intent: {rows_per_intent}")
    print(f"rows_per_split: {rows_per_split}")

    ds_filtered = df[
        df["category"].isin(categories_selected) &
        df["intent"].isin(intents_selected)
    ].rename(columns={"Unnamed: 0": "id"})
    print(f"ds_filtered: {len(ds_filtered)}")

    sampled = ds_filtered.groupby("intent", group_keys=False).sample(
        n=rows_per_intent, random_state=random_state
    )
    train_df = sampled.groupby("intent", group_keys=False).head(rows_per_split)
    test_df = sampled.drop(train_df.index)

    print(f"sampled: {len(sampled)}")
    print(f"train_df: {len(train_df)}")
    print(f"test_df: {len(test_df)}")
    return train_df, test_df

In [22]:
categories_selected = ['ACCOUNT', 'ORDER', 'PAYMENT']
intents_selected = ['create_account', 'delete_account', 'edit_account',
                    'cancel_order', 'change_order', 'place_order',
                    'check_payment_methods', 'payment_issue']

train_df, test_df = get_split_dataset(categories_selected, intents_selected, 50)

intents: 8
rows_per_intent: 6
rows_per_split: 3
ds_filtered: 7983
sampled: 48
train_df: 24
test_df: 24


In [23]:
TOP_N = 3
train_df = train_df.head(TOP_N)
train_df

,id,flags,instruction,category,intent,response
453,453,BELN,I cannot afford purchase {{Order Number}},ORDER,cancel_order,I understand the financial constraints you are...
792,792,BCELV,"I have bought some item, I have to cancel orde...",ORDER,cancel_order,I recognize that you've recently made a purcha...
209,209,BCELN,"I do not want this item, cancel order {{Order ...",ORDER,cancel_order,I've observed that you no longer want the item...


In [24]:
test_df= test_df.head(TOP_N)
test_df

,id,flags,instruction,category,intent,response
309,309,BL,I need assistance with canceling purchase {{Or...,ORDER,cancel_order,I'm attuned to the idea that you require assis...
739,739,BILQZ,were can i cancel purchase {{Order Number}},ORDER,cancel_order,"I realized, you're looking for the information..."
578,578,BQ,want assistance to cancel order {{Order Number}},ORDER,cancel_order,"I've understood, you're seeking assistance wit..."


In [25]:
from harness_agent.harness import AgentHarness

from llm import get_llm
model = get_llm("qwen/qwen3.8-27b")
# model = get_llm("openai/gpt-oss-120b")
harness = AgentHarness(model = model, training_dataset=train_df, training_mode=False, agentic_memory_disabled=True)
# response = harness.invoke("see the available payment options")
response = harness.invoke("were can i cancel purchase {{Order Number}}")

# response
print(type(response))
print(response)

Initializing AgentHarness - training_mode: False, agentic_memory_disabled: True
middlewares: 0
------harness: I am an AI assistant and do not have access to your personal account details, order history, or the specific merchant's system. Therefore, I cannot cancel the purchase for **{{Order Number}}** directly.

To cancel your order, please follow these steps:

1. **Check your confirmation email**: Look for the email you received when you made the purchase. It usually contains a link to track or manage your order.
2. **Visit the merchant’s website**: Log in to your account on the website where you made the purchase. Go to “My Orders” or “Order History” and look for an option to cancel.
3. **Contact customer support**: If you cannot cancel it yourself, reach out to the merchant’s customer service team via phone, live chat, or email. Provide them with your **{{Order Number}}** and request a cancellation.

⚠️ **Note**: Cancellation policies vary by merchant. Some orders may be non-cancell

In [26]:
from langsmith import Client
from langsmith.schemas import Dataset
from langsmith.utils import LangSmithConflictError

client = Client()

def create_dataset(name, description, delete_if_exists= False):
    try:
        if client.has_dataset(dataset_name=name):
            if delete_if_exists:
                print(f"Dataset '{name}' already exists. Deleting existing dataset...")
                client.delete_dataset(dataset_name=name)
            else:
                print(f"Dataset '{name}' already exists. Skipping and retrieving from Langsmith...")
                dataset = client.read_dataset(dataset_name=name)
                return dataset

        dataset = client.create_dataset(dataset_name = name, description = description)
        print(f"Dataset '{name}' created successfully.")
        
        return dataset       
    except LangSmithConflictError as e:
        print(f"Error on creating Dataset '{name}': {str(e)}")  

dataset_name = 'CA1 - Customer Dataset'
dataset = create_dataset(
    name = dataset_name,
    description='Customer questions and responses',
    delete_if_exists= True
)

Dataset 'CA1 - Customer Dataset' already exists. Deleting existing dataset...
Dataset 'CA1 - Customer Dataset' created successfully.


In [27]:
from pandas import DataFrame

def create_examples(dataset: Dataset, df: DataFrame, split_name: str = 'train'):
    print(f"Creating {len(df)} examples for split: {split_name}")

    example_ids = []
    for _,row in df.iterrows():
        example = client.create_example(
            dataset_id=dataset.id,            
            inputs= {
                "id": row["id"],
                "category": row["category"],
                "intent": row["intent"],
                "instruction": row["instruction"],
            },
            outputs= {
                "response": row["response"]                         
            }
        )

        example_ids.append(example.id)

    print(f"Set split name: {split_name}")
    client.update_examples(example_ids=example_ids,splits=[split_name] * len(example_ids))

In [28]:
SPLIT_TRAIN = 'train'
SPLIT_TEST = 'test'
create_examples(dataset, train_df, SPLIT_TRAIN)
create_examples(dataset, test_df, SPLIT_TEST)

Creating 3 examples for split: train
Set split name: train
Creating 3 examples for split: test
Set split name: test


In [29]:
examples = client.list_examples(dataset_name=dataset_name)
import pandas as pd

def get_examples_by_split(dataset_name, split_name):    
    examples = list(
        client.list_examples(
            dataset_name=dataset_name,
            splits=[split_name]
        )
    )

    df_examples = pd.DataFrame([
        {
            **example.inputs, #unpack operator
            **example.outputs,
            "langsmith_id": str(example.id),
            "splits": example.metadata.get("dataset_split")
                    if example.metadata else None
        }
        for example in examples
    ])

    return {
        'name' : split_name,
        'examples' : examples,
        'df': df_examples
    }

def get_examples_from_dataset(dataset_name):
    train_split = get_examples_by_split(dataset_name, SPLIT_TRAIN)
    test_split = get_examples_by_split(dataset_name, SPLIT_TEST)
    return {
        'train_split': train_split,
        'test_split': test_split,
    }

splits = get_examples_from_dataset(dataset_name)
splits

{'train_split': {'name': 'train',
  'examples': [<class 'langsmith.schemas.Example'>(id=02e1907c-dfaf-4b87-923a-7b324776a307, dataset_id=c520cc02-3379-4581-b9a0-a2d4fec8ce05, link='https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/c520cc02-3379-4581-b9a0-a2d4fec8ce05/e/02e1907c-dfaf-4b87-923a-7b324776a307'),
   <class 'langsmith.schemas.Example'>(id=9e1c49e5-42fa-422f-b7d9-9e5e68ae604d, dataset_id=c520cc02-3379-4581-b9a0-a2d4fec8ce05, link='https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/c520cc02-3379-4581-b9a0-a2d4fec8ce05/e/9e1c49e5-42fa-422f-b7d9-9e5e68ae604d'),
   <class 'langsmith.schemas.Example'>(id=daa1d910-3c42-4765-99cf-30b108770446, dataset_id=c520cc02-3379-4581-b9a0-a2d4fec8ce05, link='https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/c520cc02-3379-4581-b9a0-a2d4fec8ce05/e/daa1d910-3c42-4765-99cf-30b108770446')],
  'df':     id        intent category  \
  0  209  cancel_order    ORDER  

In [30]:
from harness_agent.harness import AgentHarness
from pydantic import BaseModel, Field
from llm import get_llm
from langchain_core.messages import HumanMessage, SystemMessage

class ResponseEvaluatorSchema(BaseModel):
    """Class to define the schema for quality evaluation."""
    score: int = Field(description="An integer score from 1 to 10 indicating the quality of the generated answer.")
    comment: str = Field(description="A brief explanation of the score")

def evaluator_agent_response_quality(inputs: dict, outputs: dict, reference_outputs: dict, model_name:str) -> dict:
    """Evaluate the quality of the agent's generated response using LLM with a structured output."""  
    query = inputs["instruction"]
    agent_response = outputs['agent_response']
    reference_response = reference_outputs['response']

    system_eval_prompt = "You are a Expert Customer Service Agent Evaluator."
    eval_prompt = f"""Given a customer query, compare and evaluate the agent's generated answer against the expected response.

    Query:
    {query}

    Expected:
    {reference_response}

    Generated:
    {agent_response}

    Rules:
        * Score from 1 to 10, where 1 is bad and 10 is excelent quality.
        * Provide a comment on the score.
        * Return structurted output.
    """

    model = get_llm(model_name)
    result = model.with_structured_output(ResponseEvaluatorSchema).invoke([SystemMessage(system_eval_prompt), HumanMessage(eval_prompt)])
    
    evaluator_result = {"score": result.score, "comment": result.comment}
    print(f"evaluator_agent_response_quality: {evaluator_result}")

    return evaluator_result

def run_agent(query:str, model_name: str, training_dataset, training_mode: bool, agentic_memory_disabled : bool):      
    model = get_llm(model_name)
    harness = AgentHarness(model = model, training_dataset=training_dataset, training_mode=training_mode, agentic_memory_disabled=agentic_memory_disabled)
    # response = harness.invoke("see the available payment options")
    response = harness.invoke(query)
    agent_response = response['agent_response']
    print("---------------------run_agent------------------------")
    print(f"response: {agent_response}")
    return agent_response

def target_function(inputs: str, model_name: str, training_dataset, training_mode: bool, agentic_memory_disabled : bool) -> dict:
    agent_response = run_agent(inputs["instruction"], model_name, training_dataset, training_mode, agentic_memory_disabled)
    print(f"----------target_function: {agent_response}")
    return {"agent_response": agent_response}


In [31]:
def evaluate_agent(model_name, training, disable_agentic_memory, split):
    dataframe = split['df']
    examples = split['examples']
    training_mode = "training" if training else "testing"
    agentic_memory_mode  = "memory_disabled" if disable_agentic_memory else "memory_on"

    print(f"Evaluate Agent --> mode: {training_mode}, agentic_memory_mode: {agentic_memory_mode}")

    client.evaluate(
        lambda inputs: target_function(inputs, model_name, dataframe, training, disable_agentic_memory),
        data= examples,
        evaluators=[lambda inputs, outputs, reference_outputs : evaluator_agent_response_quality( inputs, outputs, reference_outputs, model_name )],
        experiment_prefix=f"exp-{training_mode}-{agentic_memory_mode}-{model_name}",
        metadata={
            "model_name": model_name,
            "training_mode": training_mode,
            "agentic_memory": agentic_memory_mode,
            "split": split["name"]
        }
        )

In [32]:
#1 run in training mode with train split
#2 evaluate in test mode with test split (without agentic memory)
#3 evaluate in test mode with test split (with agentic memory)

# model_name = "openai/gpt-oss-120b"
# model_name = "deepseek"
# model_name = "gpt-6-luna"
# #1 evaluate in test mode with test split (without agentic memory)
# evaluate_agent(model_name, training=False, disable_agentic_memory=True, split=splits['test_split'])

import os
import shutil
def clean_agentic_memory():
    print("Cleaning agentic memory...")
    print(" * rmtree Skills...")
    if(os.path.exists("./skills")):
        shutil.rmtree("./skills")
    else:
        print(f"path not exist ./skills, skipping...")
    print(" * Delete AGENT.md...")
    if os.path.exists("./AGENT.md"):
        os.remove("./AGENT.md")
    else:
        print(f"path not exist ./AGENT.md, skipping...")

def evaluation_pipeline(models):
    for model in models:
        print(f"Executing evaluation pipeline for model: {model}...")   
        clean_agentic_memory()   

        evaluate_agent(model, training=False, disable_agentic_memory=True, split=splits['test_split'])
        evaluate_agent(model, training=True, disable_agentic_memory=False, split=splits['train_split'])
        evaluate_agent(model, training=False, disable_agentic_memory=False, split=splits['test_split'])

In [19]:
evaluation_pipeline([   
    # "qwen/qwen3.8-27b",
    "openai/gpt-oss-120b",
    "deepseek",
    "gpt-6-luna"]
)

Executing evaluation pipeline for model: openai/gpt-oss-120b...
Cleaning agentic memory...
 * rmtree Skills...
path not exist ./skills, skipping...
 * Delete AGENT.md...
path not exist ./AGENT.md, skipping...


NameError: name 'evaluate_agent' is not defined

In [51]:
#2 run in training mode with train split
evaluate_agent(model_name, training_mode=True, disable_agentic_memory=False, split=splits['train_split'])

View the evaluation results for experiment: 'exp-ce54f689' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/1b7e0dbd-abb8-4fa8-ad25-1ebd5026077a/compare?selectedSessions=5f95581f-64da-4349-af3e-084fae9bf872


Initializing AgentHarness - training_mode: True, agentic_memory_disabled: False
middlewares: 2
before_agent 1 messages
before_agent 1 messages
request_supervisor: see the available payment options, refeerence: Example response: Thank you for your interest in exploring our available payment options! We take pride in offering a variety of payment methods to cater to our customers' needs. To view the available payment options, you can visit our website and navigate to the payment page. There, you will find a comprehensive list of the payment methods we accept, including credit/debit cards, PayPal, bank transfers, Apple Pay, and Google Wallet. If you have any questions or need further assistance with any of these options, please don't hesitate to reac

In [52]:
#3 evaluate in test mode with test split (with agentic memory)
evaluate_agent(model_name, training_mode=False, disable_agentic_memory=False, split=splits['test_split'])

View the evaluation results for experiment: 'exp-f5c5ba6a' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/1b7e0dbd-abb8-4fa8-ad25-1ebd5026077a/compare?selectedSessions=b34f055f-809d-4e92-8736-ee1c858ef4e0


Initializing AgentHarness - training_mode: False, agentic_memory_disabled: False
middlewares: 2
before_agent 1 messages
before_agent 1 messages
------harness: Thank you for your interest in our payment options! We offer a variety of methods to suit your needs, including credit/debit cards, PayPal, bank transfers, Apple Pay, and Google Wallet.

You can view the full list of accepted payment methods on our website by going to the Payment page.

If you have any questions or need help with a particular option, just let us know – we’re here to make the process as smooth as possible!
---------------------run_agent------------------------
response: Thank you for your interest in our payment options! We offer a variety of methods to suit your needs, inclu